# PHYT1D — Module 03: VO2max and VO2rest Estimation

Prior estimates of aerobic capacity from age and sex using the LowLands Fitness Registry equations.

**Reference:** Van de Poppe 2021 [21]; Byrne 2005 [22]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─── T1DS age-group mapping ───────────────────────────────────────────────────
AGE_GROUP_MAP = {
    "child"      : 10,   # years
    "adolescent" : 15,
    "adult"      : 35,   # default
}


## 3.1 VO2max Prior Estimate (Van de Poppe 2021)

In [ ]:
def estimate_VO2max(age_or_group, sex):
    """
    VO2max prior from LowLands Fitness Registry quadratic equations.

    Males   : VO2max = -0.0010·age² − 0.2248·age + 54.286  [mL/kg/min]
    Females : VO2max = -0.0021·age² − 0.1407·age + 43.066  [mL/kg/min]

    Parameters
    ----------
    age_or_group : int | str — actual age in years OR 'child'/'adolescent'/'adult'
    sex          : str       — 'M' | 'F'

    Returns
    -------
    float — prior VO2max estimate [mL/kg/min]
    """
    if isinstance(age_or_group, str):
        age = AGE_GROUP_MAP[age_or_group.lower()]
    else:
        age = float(age_or_group)

    if sex.upper() == "M":
        vo2 = -0.0010 * age**2 - 0.2248 * age + 54.286
    elif sex.upper() == "F":
        vo2 = -0.0021 * age**2 - 0.1407 * age + 43.066
    else:
        raise ValueError(f"sex must be 'M' or 'F', got '{sex}'")

    return max(vo2, 15.0)   # physiological floor


def VO2max_prior(age_or_group, sex, sigma=8.0):
    """
    Returns (mean, std) for the MCMC Window B VO2max prior.
    Prior: N(VO2max_age_sex, 8²)
    """
    mean = estimate_VO2max(age_or_group, sex)
    return mean, sigma


## 3.2 VO2rest Estimation

In [ ]:
def estimate_VO2rest(age_or_group):
    """
    Age-specific resting VO2. NOT fixed at 3.5 mL/kg/min.
    Used when converting from absolute workload metrics.

    Approximate values from Byrne 2005 [22] / Kwan 2004 [23]:
      Child (~10 yr)     : ~4.5 mL/kg/min
      Adolescent (~15 yr): ~4.0 mL/kg/min
      Adult (~35 yr)     : ~3.5 mL/kg/min

    Parameters
    ----------
    age_or_group : int | str

    Returns
    -------
    float — VO2rest [mL/kg/min]
    """
    lookup = {"child": 4.5, "adolescent": 4.0, "adult": 3.5}
    if isinstance(age_or_group, str):
        return lookup.get(age_or_group.lower(), 3.5)
    age = float(age_or_group)
    if age < 13:   return 4.5
    elif age < 20: return 4.0
    else:          return 3.5


## 3.3 Reproduce Figure 8 — VO2max vs Age

In [ ]:
ages = np.arange(10, 55, 0.5)
vo2_M = [-0.0010*a**2 - 0.2248*a + 54.286 for a in ages]
vo2_F = [-0.0021*a**2 - 0.1407*a + 43.066 for a in ages]

plt.figure(figsize=(9, 5))
plt.plot(ages, vo2_M, "b-",  lw=2, label="Male")
plt.plot(ages, vo2_F, "b--", lw=2, label="Female")

# Annotate T1DS age groups
for grp, col in [("child", "green"), ("adolescent", "orange"), ("adult", "red")]:
    age = AGE_GROUP_MAP[grp]
    m = estimate_VO2max(age, "M")
    f = estimate_VO2max(age, "F")
    plt.scatter([age], [m], color=col, zorder=5, s=70)
    plt.scatter([age], [f], color=col, zorder=5, s=70, marker="^")
    plt.annotate(f"{grp.capitalize()} ({age} yr)
M:{m:.1f} F:{f:.1f}",
                 xy=(age, (m+f)/2), xytext=(age+1, (m+f)/2+1.5),
                 fontsize=8, arrowprops=dict(arrowstyle="->", color="grey"))

plt.xlabel("Age (years)", fontsize=12)
plt.ylabel("VO₂max (mL/kg/min)", fontsize=12)
plt.title("Figure 8 — VO₂max Estimation (LowLands Fitness Registry)", fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("vo2max_estimation.png", dpi=150)
plt.show()

# Print prior summary
print("\n── VO2max Prior Summary (mean ± 1σ, σ=8 mL/kg/min) ─────────────────")
for grp in ["child", "adolescent", "adult"]:
    for sex in ["M", "F"]:
        m, s = VO2max_prior(grp, sex)
        print(f"  {grp:12s} {sex}:  {m:.1f} ± {s:.1f} mL/kg/min")
